# ViBioMIR 2026 - Experiment A1: Qwen3-Embedding-0.6B + RRF Fusion

This notebook runs the GPU Dense Embedding and Retrieval pipeline on Kaggle:
1. **Input**: Candidate chunks (103k docs, ~467k chunks) pre-computed locally.
2. **Model**: `Qwen/Qwen3-Embedding-0.6B` with `float16` precision and batch size 64.
3. **Dense Index**: Exact Inner Product (`IndexFlatIP`) with FAISS.
4. **Dense Search**: Retrieve top-500 candidate chunks for 1,200 competition queries.
5. **RRF Fusion**: Merge BM25 Top 1,000 and Dense Top 1,000.
6. **Submission**: Export & Validate official `submission.json` and `submission.zip`.
7. **Vector Artifacts**: Zip all vector embeddings and index for download back to local machine.

> **Prerequisites on Kaggle**:
> - **Accelerator**: GPU T4 x2 or GPU P100 (16GB VRAM)
> - **Internet**: ON (to download model weights from HuggingFace)

In [ ]:
# 1. Check GPU Device
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: No GPU detected! Please enable GPU accelerator in Notebook Settings.")

In [ ]:
# 2. Locate Input Dataset Paths
import os, glob
from pathlib import Path

# Search for the dataset in /kaggle/input
input_dir = None
for p in glob.glob("/kaggle/input/**/retrieval_top_1000.parquet", recursive=True):
    input_dir = Path(p).parent
    break

if input_dir is None:
    # Fallback to direct check
    candidates = list(Path("/kaggle/input").glob("*"))
    raise FileNotFoundError(
        f"Could not find retrieval_top_1000.parquet! Available in /kaggle/input: {candidates}. "
        "Please make sure you have added your uploaded dataset to this notebook."
    )

chunks_dir = input_dir / "chunks"
bm25_file = input_dir / "retrieval_top_1000.parquet"
query_file = input_dir / "query.parquet"
wheels = list(input_dir.glob("*.whl"))

print(f"Dataset Directory : {input_dir}")
print(f"Chunks Directory  : {chunks_dir} (exists: {chunks_dir.exists()})")
print(f"BM25 Hits File    : {bm25_file} (exists: {bm25_file.exists()})")
print(f"Query Parquet     : {query_file} (exists: {query_file.exists()})")
if wheels:
    print(f"Found Package Wheel: {wheels[0]}")

In [ ]:
# 3. Install Required Packages
if wheels:
    !pip install -q --no-deps {wheels[0]}
!pip install -q tantivy faiss-cpu sentence-transformers transformers pyyaml pyarrow
print("Environment dependencies installed successfully!")

In [ ]:
# 4. Generate Working Experiment Configuration
import yaml

work_dir = Path("/kaggle/working/exp_a1")
work_dir.mkdir(parents=True, exist_ok=True)
config_file = work_dir / "config_kaggle.yaml"

config_data = {
    "experiment_id": "exp_a1",
    "description": "ViBioMIR 2026 Experiment A1 on Kaggle GPU",
    "chunking": {
        "strategy": "sentence_aware",
        "target_tokens": 400,
        "max_total_tokens": 512,
        "overlap_tokens": 48,
        "chunking_version": "a1",
    },
    "model": {
        "model_id": "Qwen/Qwen3-Embedding-0.6B",
        "local_model_path": None,
        "offline_mode": False,
        "device": "cuda",
        "precision": "float16",
        "batch_size": 64,
        "max_length": 512,
        "query_instruction": "Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery: ",
        "pooling_method": "last_token",
        "normalize": True,
    },
    "index": {
        "index_type": "flat_ip",
        "use_faiss": True,
        "metric": "cosine",
    },
    "retrieval": {
        "dense_chunk_top_k": 500,
        "document_aggregation": "max",
        "top_k_docs": 1000,
    },
    "fusion": {
        "rrf_k": 60,
        "bm25_weight": 1.0,
        "dense_weight": 1.0,
        "top_k": 1000,
    },
    "paths": {
        "candidate_doc_ids": "",
        "bm25_hits": str(bm25_file),
        "queries": str(query_file),
        "output_dir": str(work_dir),
    },
    "submission": {
        "top_docs": 10,
        "top_chunks": 10,
        "chunks_per_doc": 1,
        "include_chunk_order": False,
        "doc_id_type": "int",
        "output_filename": "submission.json",
        "zip_filename": "submission.zip",
    }
}

with open(config_file, "w", encoding="utf-8") as f:
    yaml.dump(config_data, f, default_flow_style=False)

print(f"Configuration saved to: {config_file}")

In [ ]:
# 5. Step 1: Run Dense Embedding on Chunks (approx. 10-15 mins)
!vibiomir-retrieval embed \
    --config /kaggle/working/exp_a1/config_kaggle.yaml \
    --chunks-dir {chunks_dir} \
    --batch-size 64 \
    --precision float16 \
    --device cuda

In [ ]:
# 6. Step 2: Build FAISS Dense Vector Index
!vibiomir-retrieval build-dense-index \
    --config /kaggle/working/exp_a1/config_kaggle.yaml

In [ ]:
# 7. Step 3: Run Dense Search on 1,200 Queries
!vibiomir-retrieval dense-search \
    --config /kaggle/working/exp_a1/config_kaggle.yaml \
    --query-path {query_file} \
    --device cuda

In [ ]:
# 8. Step 4: Reciprocal Rank Fusion (BM25 + Dense)
!vibiomir-retrieval fuse \
    --config /kaggle/working/exp_a1/config_kaggle.yaml \
    --bm25-file {bm25_file}

In [ ]:
# 9. Step 5: Export Official ViBioMIR Submission (JSON + ZIP)
!vibiomir-retrieval export-submission \
    --config /kaggle/working/exp_a1/config_kaggle.yaml \
    --chunks-dir {chunks_dir} \
    --query-path {query_file} \
    --top-docs 10 \
    --top-chunks 10

In [ ]:
# 10. Step 6: Validate Submission Format
!vibiomir-retrieval validate-submission \
    --submission /kaggle/working/exp_a1/submission/submission.zip \
    --query-path {query_file}

In [ ]:
# 11. Step 7: Package Vector Embeddings & Outputs for Download to Local
import os, shutil
from pathlib import Path

work_dir = Path("/kaggle/working")
sub_zip = work_dir / "exp_a1/submission/submission.zip"
if sub_zip.exists():
    shutil.copy(sub_zip, work_dir / "submission.zip")
    print("Copied submission.zip to root output directory!")

!cd /kaggle/working && zip -r -q vibiomir_vectors_exp_a1.zip exp_a1/embeddings exp_a1/index exp_a1/retrieval exp_a1/fusion

print("=== FINISHED ALL TASKS! ===")
print("Outputs created in /kaggle/working/:")
!ls -lh /kaggle/working/*.zip